# Customer Churn Exploratory Data Analysis

This notebook explores the Telco customer churn dataset before modeling. It creates a stratified train/test split immediately after loading the file, then uses only the training set for EDA so the test set remains untouched for final model evaluation.

## 1. Import Libraries


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

sns.set_theme(style="whitegrid", palette="Set2")

## 2. Load Dataset and Create Train/Test Split

In [ ]:
DATA_PATH = Path("data/Telco_customer_churn.xlsx")

TEST_SIZE = 0.20
RANDOM_STATE = 42
TARGET_COLUMN = "Churn Value"

df = pd.read_excel(DATA_PATH)

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COLUMN],
)

# Use only the training data for EDA-driven modeling decisions.
eda_df = train_df.copy()

split_summary = pd.DataFrame(
    {
        "rows": [len(df), len(train_df), len(test_df)],
        "churn_rate": [
            df[TARGET_COLUMN].mean(),
            train_df[TARGET_COLUMN].mean(),
            test_df[TARGET_COLUMN].mean(),
        ],
    },
    index=["full_data", "train", "test"],
)

display(split_summary)
eda_df.head()

## 3. Training Set Overview

In [ ]:
print(f"Training rows: {eda_df.shape[0]:,}")
print(f"Training columns: {eda_df.shape[1]:,}")

eda_df.info()

In [ ]:
eda_df.describe(include="all").T

## 4. Training Set Missing Values and Duplicates

In [ ]:
missing_summary = (
    eda_df.isna().sum()
    .to_frame("missing_count")
    .assign(missing_percent=lambda x: x["missing_count"] / len(eda_df) * 100)
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

missing_summary

In [ ]:
blank_counts = (
    eda_df.select_dtypes(include="object")
    .apply(lambda col: col.astype(str).str.strip().eq("").sum())
    .sort_values(ascending=False)
)

blank_counts[blank_counts > 0]

In [ ]:
print(f"Duplicate training rows: {eda_df.duplicated().sum():,}")
print(f"Duplicate training customer IDs: {eda_df['CustomerID'].duplicated().sum():,}")

## 5. Target Variable: Churn in the Training Set

In [ ]:
churn_counts = eda_df["Churn Label"].value_counts()
churn_rate = eda_df["Churn Value"].mean()

print(churn_counts)
print(f"\nTraining churn rate: {churn_rate:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=eda_df, x="Churn Label", ax=ax)
ax.set_title("Training Set Customer Churn Distribution")
ax.set_xlabel("Churn")
ax.set_ylabel("Number of Customers")
plt.show()

## 6. Numeric Feature Distributions


In [ ]:
numeric_columns = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "Churn Score",
    "CLTV",
]

for column in numeric_columns:
    eda_df[column] = pd.to_numeric(eda_df[column], errors="coerce")

eda_df[numeric_columns].describe().T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, column in zip(axes, numeric_columns):
    sns.histplot(data=eda_df, x=column, hue="Churn Label", kde=True, ax=ax)
    ax.set_title(f"{column} by Churn")

for ax in axes[len(numeric_columns):]:
    ax.remove()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, column in zip(axes, numeric_columns):
    sns.histplot(
        data=eda_df,
        x=column,
        hue="Churn Label",
        multiple="fill",
        bins=20,
        ax=ax,
    )
    ax.set_title(f"Churn Share by {column}")
    ax.set_ylabel("Share within bin")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))

for ax in axes[len(numeric_columns):]:
    ax.remove()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, column in zip(axes, ["Tenure Months", "Monthly Charges", "Total Charges"]):
    sns.boxplot(data=eda_df, x="Churn Label", y=column, ax=ax)
    ax.set_title(f"{column} vs Churn")

plt.tight_layout()
plt.show()

## 7. Churn by Categorical Features


In [ ]:
def plot_churn_rate(column, top_n=None):
    data = eda_df.copy()
    if top_n is not None:
        top_values = data[column].value_counts().head(top_n).index
        data = data[data[column].isin(top_values)]

    churn_by_group = (
        data.groupby(column, dropna=False)["Churn Value"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=churn_by_group, x=column, y="Churn Value", ax=ax)
    ax.set_title(f"Training Churn Rate by {column}")
    ax.set_xlabel(column)
    ax.set_ylabel("Churn Rate")
    ax.tick_params(axis="x", rotation=35)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
    plt.tight_layout()
    plt.show()

    return churn_by_group

In [ ]:
for column in [
    "Contract",
    "Internet Service",
    "Payment Method",
    "Paperless Billing",
    "Senior Citizen",
    "Dependents",
    "Partner",
]:
    display(plot_churn_rate(column))


## 8. Training Set Churn Reasons

This section looks only at training-set customers who churned. `Churn Reason` should not be used as a model feature because it is only known after churn happens.

In [ ]:
churned = eda_df[eda_df["Churn Value"] == 1]

reason_counts = churned["Churn Reason"].value_counts().head(15)
reason_counts

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=reason_counts.values, y=reason_counts.index, ax=ax)
ax.set_title("Top Churn Reasons")
ax.set_xlabel("Number of Churned Customers")
ax.set_ylabel("Churn Reason")
plt.tight_layout()
plt.show()


## 9. Correlation Between Numeric Features


In [ ]:
corr_columns = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "Churn Score",
    "CLTV",
    "Churn Value",
]

corr = eda_df[corr_columns].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f", ax=ax)
ax.set_title("Training Set Numeric Feature Correlation")
plt.tight_layout()
plt.show()

## 10. Initial EDA Takeaways

Use this section to write observations after running the notebook. These observations come from the training data only, so the test set remains a clean final evaluation set.

Useful points to check:

- Is churn concentrated in month-to-month contracts?
- Do newer customers churn more often than long-tenure customers?
- Are churn rates higher for specific internet services or payment methods?
- Which fields are leakage columns and should be excluded from modeling?

In [ ]:
summary = {
    "training_customers": len(eda_df),
    "training_churned_customers": int(eda_df["Churn Value"].sum()),
    "training_churn_rate": eda_df["Churn Value"].mean(),
    "training_average_monthly_charges": eda_df["Monthly Charges"].mean(),
    "training_median_tenure_months": eda_df["Tenure Months"].median(),
    "held_out_test_customers": len(test_df),
}

pd.Series(summary)